<a href="https://colab.research.google.com/github/nadynstary/DR-using-DINOv2-and-Kmeans-self-supervised-/blob/main/Dimensionality_Reduction_(selfsupervised).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import urllib.request
import tarfile

url = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
urllib.request.urlretrieve(url, "images.tar")

with tarfile.open("images.tar") as tar:
    tar.extractall("./stanford_dogs")

/tmp/ipykernel_1065/3717473315.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("./stanford_dogs")


In [2]:
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms

dataset = ImageFolder(
    root="./stanford_dogs/Images",
    transform=transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
    ])
)
print(dataset.classes)  #breed names (120)

['n02085620-Chihuahua', 'n02085782-Japanese_spaniel', 'n02085936-Maltese_dog', 'n02086079-Pekinese', 'n02086240-Shih-Tzu', 'n02086646-Blenheim_spaniel', 'n02086910-papillon', 'n02087046-toy_terrier', 'n02087394-Rhodesian_ridgeback', 'n02088094-Afghan_hound', 'n02088238-basset', 'n02088364-beagle', 'n02088466-bloodhound', 'n02088632-bluetick', 'n02089078-black-and-tan_coonhound', 'n02089867-Walker_hound', 'n02089973-English_foxhound', 'n02090379-redbone', 'n02090622-borzoi', 'n02090721-Irish_wolfhound', 'n02091032-Italian_greyhound', 'n02091134-whippet', 'n02091244-Ibizan_hound', 'n02091467-Norwegian_elkhound', 'n02091635-otterhound', 'n02091831-Saluki', 'n02092002-Scottish_deerhound', 'n02092339-Weimaraner', 'n02093256-Staffordshire_bullterrier', 'n02093428-American_Staffordshire_terrier', 'n02093647-Bedlington_terrier', 'n02093754-Border_terrier', 'n02093859-Kerry_blue_terrier', 'n02093991-Irish_terrier', 'n02094114-Norfolk_terrier', 'n02094258-Norwich_terrier', 'n02094433-Yorkshire_t

importing methods

In [3]:
import torch
from PIL import Image
import torchvision.transforms as T
import numpy as np
from sklearn.cluster import KMeans

In [4]:
device= "cuda" if torch.cuda.is_available() else "cpu"

# loading the DINOv2
dinov2= torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov2.eval().to(device)

# preprocessing & normalization
transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229,0.224,0.229])
])

image_paths = [path for path, _ in dataset.samples]
class_indices = [label for _, label in dataset.samples]
class_labels = [dataset.classes[i] for i in class_indices]


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 306MB/s]


In [5]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def get_embeddings_batched(image_paths, batch_size=32):
    all_embeddings = []

    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]

        # Load and transform a whole batch at once
        batch_imgs = []
        for path in batch_paths:
            img = Image.open(path).convert("RGB")
            batch_imgs.append(transform(img))

        batch_tensor = torch.stack(batch_imgs).to(device)  # shape: [batch_size, 3, 224, 224]

        with torch.no_grad():
            batch_embeddings = dinov2(batch_tensor)  # shape: [batch_size, 768]

        all_embeddings.append(batch_embeddings.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)

embeddings = get_embeddings_batched(image_paths, batch_size=32)
np.save("dinov2_embeddings.npy", embeddings)
np.save("class_labels.npy", class_labels)

# Later, just reload instead of recomputing:
embeddings = np.load("dinov2_embeddings.npy")
class_labels = np.load("class_labels.npy", allow_pickle=True)

100%|██████████| 644/644 [06:56<00:00,  1.55it/s]


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import numpy as np
np.save("/content/drive/MyDrive/dinov2_embeddings.npy", embeddings)
np.save("/content/drive/MyDrive/class_labels.npy", class_labels)

print("DONE!")
print("embeddings shape:", embeddings.shape)

DONE!
embeddings shape: (20580, 768)
